In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
import statsmodels.api as sm
from scipy.stats import norm

# Configurações para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

Escolhendo as covariáveis com base na análise exploratória de dados, e dos conhecimentos da primeira parte do trabalho. Continuamos com a variávei de trend, adicionamos as duas covariáveis (`inv` e `users`) adquiridas para a segunda parte, elas com seus lags e suas versões logaritmicas, para a previsão logarítmica da variável `volume`.

In [2]:
# Carregando os dados originais
data = pd.read_csv("./data_updated.csv")
data["week"] = pd.to_datetime(data["week"])
data.set_index("week", inplace=True)

data["inv"] = data["inv"].shift(1)
data["users"] = data["users"].shift(1)

data["log_inv"] = np.log(data["inv"])
data["log_users"] = np.log(data["users"])

data["users_lag_8"] = data["users"].shift(7)

data["users_log_lag_8"] = np.log(data["users_lag_8"])

data["trend"] = np.arange(len(data))

data.dropna(inplace=True)

In [3]:
data.head()

,volume,inv,users,log_inv,log_users,users_lag_8,users_log_lag_8,trend
week,,,,,,,,
2022-12-26,0.39,0.182806,0.655,-1.699331,-0.423120,6.500,1.871802,8
2023-01-02,0.33,0.182695,0.778,-1.699940,-0.251029,7.061,1.954587,9
2023-01-09,0.36,0.121898,0.808,-2.104570,-0.213193,5.875,1.770706,10
2023-01-16,0.49,0.361241,1.641,-1.018209,0.495306,24.238,3.187922,11
2023-01-23,0.50,0.448260,2.323,-0.802381,0.842859,7.648,2.034444,12


In [4]:
X = data.drop('volume', axis=1)
y = data['volume']

Definindo os modelos de regressão linear com as covariáveis selecionadas, neste caso usamos as covariáveis `trend`, `inv` e `users`, suas versões logarítmicas e seus lags.

In [5]:
predictions = {}
models = {}

models_cols = {
    "Covars": ["inv", "users", "users_lag_8"],
    "Covars + Trend": ["trend", "inv", "users", "users_lag_8"],
    "Log Covars + Trend": ["trend", "log_inv", "log_users", "users_log_lag_8"],
}

In [6]:
results = {}

for model_name, cols in models_cols.items():
    X_model = sm.add_constant(X[cols])

    model = sm.OLS(y, X_model).fit()
    models[model_name] = model

    metrics = {}

    metrics["Adjusted R²"] = models[model_name].rsquared_adj
    metrics["AIC"] = models[model_name].aic
    metrics["BIC"] = models[model_name].bic
    metrics["RMSE"] = np.sqrt(models[model_name].mse_resid)

    results[model_name] = metrics

results_df = pd.DataFrame(results).T
display(results_df.sort_values(by="Adjusted R²", ascending=False))

,Adjusted R²,AIC,BIC,RMSE
Log Covars + Trend,0.837868,648.324874,663.344606,2.096266
Covars + Trend,0.768132,701.630833,716.650565,2.506875
Covars,0.287820,867.864353,879.880139,4.393467


Agora analisando os dois melhores modelos para verificar se seus parâmetros são estatisticamente significativos.

In [7]:
models["Log Covars + Trend"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.842
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     192.2
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           1.11e-56
Time:                        14:57:28   Log-Likelihood:                -319.16
No. Observations:                 149   AIC:                             648.3
Df Residuals:                     144   BIC:                             663.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.6318      0.556     -1.137      0.258      -1.730       0.467
trend               0.1039      0.005     19.444      0.000       0.093       0.114
log_inv             1.5704      0.209      7.528      0.000       1.158       1.983
log_users          -1.5526      0.340     -4.566      0.000      -2.225      -0.881
users_log_lag_8     0.9359      0.190      4.916      0.000       0.560       1.312
==============================================================================
Omnibus:                       75.443   Durbin-Watson:                   0.619
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              326.422
Skew:                           1.870   Prob(JB):                     1.31e-71
Kurtosis:                       9.212   Cond. No.                         337.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Todos os p-valores pequenos e intervalos de confianção fora do 0, indicam que os parâmetros são estatisticamente significativos.

In [8]:
models["Covars + Trend"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.774
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     123.6
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           1.57e-45
Time:                        14:57:28   Log-Likelihood:                -345.82
No. Observations:                 149   AIC:                             701.6
Df Residuals:                     144   BIC:                             716.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -4.2004      0.467     -8.997      0.000      -5.123      -3.278
trend           0.1023      0.006     17.360      0.000       0.091       0.114
inv             2.4589      0.604      4.070      0.000       1.265       3.653
users          -0.1804      0.073     -2.476      0.014      -0.324      -0.036
users_lag_8     0.0660      0.040      1.664      0.098      -0.012       0.144
==============================================================================
Omnibus:                       51.676   Durbin-Watson:                   0.385
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              146.442
Skew:                           1.368   Prob(JB):                     1.59e-32
Kurtosis:                       7.013   Cond. No.                         287.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Neste caso o users_lag_8 não aparenta ser estatisticamente significativo por pouco, com um p-valor de 0.09 e um intervalo de confiança que inclui o zero.

Vamos fazer a nova versão do Covars + Trend, removendo o users_lag_8.

In [9]:
models_cols.update({
    "Covars + Trend (2)": ["trend", "inv", "users"],
})

results = {}

for model_name, cols in models_cols.items():
    X_model = sm.add_constant(X[cols])

    model = sm.OLS(y, X_model).fit()
    models[model_name] = model

    metrics = {}

    metrics["Adjusted R²"] = models[model_name].rsquared_adj
    metrics["AIC"] = models[model_name].aic
    metrics["BIC"] = models[model_name].bic
    metrics["RMSE"] = np.sqrt(models[model_name].mse_resid)

    results[model_name] = metrics

results_df = pd.DataFrame(results).T
display(results_df.sort_values(by="Adjusted R²", ascending=False))

,Adjusted R²,AIC,BIC,RMSE
Log Covars + Trend,0.837868,648.324874,663.344606,2.096266
Covars + Trend,0.768132,701.630833,716.650565,2.506875
Covars + Trend (2),0.765302,702.469429,714.485214,2.522126
Covars,0.287820,867.864353,879.880139,4.393467


In [10]:
models["Covars + Trend (2)"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.770
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     161.9
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           4.43e-46
Time:                        14:57:28   Log-Likelihood:                -347.23
No. Observations:                 149   AIC:                             702.5
Df Residuals:                     145   BIC:                             714.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.1927      0.470     -8.927      0.000      -5.121      -3.264
trend          0.1051      0.006     18.523      0.000       0.094       0.116
inv            2.8089      0.570      4.929      0.000       1.683       3.935
users         -0.1866      0.073     -2.547      0.012      -0.331      -0.042
==============================================================================
Omnibus:                       44.382   Durbin-Watson:                   0.414
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              112.863
Skew:                           1.206   Prob(JB):                     3.11e-25
Kurtosis:                       6.516   Cond. No.                         274.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Com isto obtemos um modelo mais estatisticamente significativo, com todos os p-valores pequenos e intervalos de confiança fora do 0.

### Análise dos modelos de regressão linear para a escala log(volume)

In [11]:
y_log = np.log(y)

Agora vamos criar os modelos para prever log(volume) ao inves de volume diretamente. Começaremos definindo novamente os modelos.

In [12]:
log_models = {}

predictions = {}
models = {}

models_cols = {
    "Covars": ["inv", "users", "users_lag_8"],
    "Covars + Trend": ["trend", "inv", "users", "users_lag_8"],
    "Log Covars + Trend": ["trend", "log_inv", "log_users", "users_log_lag_8"],
}

results = {}

for model_name, cols in models_cols.items():
    X_model = sm.add_constant(X[cols])

    model = sm.OLS(y_log, X_model).fit()
    log_models[model_name] = model

    metrics = {}

    metrics["Adjusted R²"] = log_models[model_name].rsquared_adj
    metrics["AIC"] = log_models[model_name].aic
    metrics["BIC"] = log_models[model_name].bic

    results[model_name] = metrics

results_df = pd.DataFrame(results).T
display(results_df.sort_values(by="Adjusted R²", ascending=False))

,Adjusted R²,AIC,BIC
Log Covars + Trend,0.961015,13.709727,28.729459
Covars + Trend,0.949814,51.339986,66.359717
Covars,0.509885,389.928940,401.944726


Analisaremos os dois melhores modelos para verificar se seus parâmetros são estatisticamente significativos.

In [13]:
log_models["Log Covars + Trend"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.962
Model:                            OLS   Adj. R-squared:                  0.961
Method:                 Least Squares   F-statistic:                     913.1
Date:                Sun, 30 Nov 2025   Prob (F-statistic):          3.42e-101
Time:                        14:57:28   Log-Likelihood:                -1.8549
No. Observations:                 149   AIC:                             13.71
Df Residuals:                     144   BIC:                             28.73
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -1.3894      0.066    -21.026      0.000      -1.520      -1.259
trend               0.0210      0.001     33.077      0.000       0.020       0.022
log_inv            -0.0119      0.025     -0.479      0.633      -0.061       0.037
log_users           0.2770      0.040      6.853      0.000       0.197       0.357
users_log_lag_8     0.2050      0.023      9.060      0.000       0.160       0.250
==============================================================================
Omnibus:                       14.538   Durbin-Watson:                   0.750
Prob(Omnibus):                  0.001   Jarque-Bera (JB):                4.935
Skew:                           0.010   Prob(JB):                       0.0848
Kurtosis:                       2.109   Cond. No.                         337.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Neste caso log_inv não aparenta ser estatisticamente significativo, com um p-valor de 0.6 e um intervalo de confiança que inclui o zero.

In [14]:
log_models["Covars + Trend"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.950
Method:                 Least Squares   F-statistic:                     701.3
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           2.67e-93
Time:                        14:57:28   Log-Likelihood:                -20.670
No. Observations:                 149   AIC:                             51.34
Df Residuals:                     144   BIC:                             66.36
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.4363      0.053    -27.275      0.000      -1.540      -1.332
trend           0.0237      0.001     35.666      0.000       0.022       0.025
inv             0.1949      0.068      2.860      0.005       0.060       0.330
users           0.0233      0.008      2.829      0.005       0.007       0.040
users_lag_8     0.0271      0.004      6.063      0.000       0.018       0.036
==============================================================================
Omnibus:                        3.772   Durbin-Watson:                   0.740
Prob(Omnibus):                  0.152   Jarque-Bera (JB):                2.381
Skew:                          -0.083   Prob(JB):                        0.304
Kurtosis:                       2.403   Cond. No.                         287.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Aqui iremos atualizar o modelo Log Covars + Trend removendo log_inv.

In [15]:
log_models = {}

predictions = {}
models = {}

models_cols.update({
    "Log Covars + Trend (2)": ["trend", "log_users", "users_log_lag_8"],
})

results = {}

for model_name, cols in models_cols.items():
    X_model = sm.add_constant(X[cols])

    model = sm.OLS(y_log, X_model).fit()
    log_models[model_name] = model

    metrics = {}

    metrics["Adjusted R²"] = log_models[model_name].rsquared_adj
    metrics["AIC"] = log_models[model_name].aic
    metrics["BIC"] = log_models[model_name].bic

    results[model_name] = metrics

results_df = pd.DataFrame(results).T
display(results_df.sort_values(by="Adjusted R²", ascending=False))

,Adjusted R²,AIC,BIC
Log Covars + Trend (2),0.961222,11.946457,23.962242
Log Covars + Trend,0.961015,13.709727,28.729459
Covars + Trend,0.949814,51.339986,66.359717
Covars,0.509885,389.928940,401.944726


In [16]:
log_models["Log Covars + Trend (2)"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 volume   R-squared:                       0.962
Model:                            OLS   Adj. R-squared:                  0.961
Method:                 Least Squares   F-statistic:                     1224.
Date:                Sun, 30 Nov 2025   Prob (F-statistic):          1.01e-102
Time:                        14:57:28   Log-Likelihood:                -1.9732
No. Observations:                 149   AIC:                             11.95
Df Residuals:                     145   BIC:                             23.96
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -1.3660      0.044    -30.766      0.000      -1.454      -1.278
trend               0.0212      0.001     38.492      0.000       0.020       0.022
log_users           0.2614      0.024     10.965      0.000       0.214       0.309
users_log_lag_8     0.2047      0.023      9.074      0.000       0.160       0.249
==============================================================================
Omnibus:                       14.648   Durbin-Watson:                   0.736
Prob(Omnibus):                  0.001   Jarque-Bera (JB):                4.956
Skew:                           0.014   Prob(JB):                       0.0839
Kurtosis:                       2.107   Cond. No.                         203.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

O novo modelo apresentou melhor desempenho nas métricas de avaliação se comparado ao anterior, além disso todos os parâmetros são estatisticamente significativos. Além disso se mostrou melhor que todos os outros modelos analisados, mostrando como as melhores variáveis para explicar log(volume) são trend, log(users) e log(users_lag_8).